In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("/notebooks/"))
sys.path.insert(0, os.path.abspath("../"))

import torch
from pathlib import Path
from ml.events import (
    INSTR_NAMES, TIME_SIGS, TEMPO_BUCKETS,
    sos_event, eos_event, bar_event, key_event, time_event, tempo_event,
)
from ml.tokenizer import events_to_score, count_bars
from ml.model import load_model, CHECKPOINT_DIR

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

print("Available instruments:")
for i, name in enumerate(INSTR_NAMES):
    print(f"  {i:2d}: {name}")

### Generation Parameters

In [ ]:
# Instruments to include in the generated score (names from INSTR_NAMES above)
INSTRUMENTS = ["voice", "piano"]

# Number of measures to generate
N_MEASURES = 24

# Key signature: number of sharps (-6=6 flats, 0=C major, 6=6 sharps)
KEY_SHARPS = 0      # 0 = C major / A minor

# Time signature index into TIME_SIGS = [2/4, 3/4, 4/4, 6/8, 3/8, 12/8]
TIME_SIG_IDX = 2    # 2 = 4/4

# Tempo bucket index into TEMPO_BUCKETS ([20,40,52,...,256] BPM boundaries)
TEMPO_IDX = 5

# Sampling parameters
TEMPERATURE = 0.9
TOP_K = 50
MAX_NEW = 1500
MODEL_NAME = "MMT-GAN-D512-L6"   # or MMT-GEN-D512-L6 for the pre-GAN model

### Generate

In [ ]:
# Resolve instrument indices
instr_indices = []
for name in INSTRUMENTS:
    name_lower = name.lower()
    if name_lower in INSTR_NAMES:
        instr_indices.append(INSTR_NAMES.index(name_lower))
    else:
        print(f"Unknown instrument '{name}', falling back to 'other'")
        instr_indices.append(0)

print(f"Loading model {MODEL_NAME}...")
model, cfg = load_model(MODEL_NAME, DEVICE)
print(f"Model loaded ({model.num_params():,} params)")

# Header: one INSTR event per part; the model samples each part's clef.
header = model.sample_header(instr_indices, TEMPERATURE, TOP_K, DEVICE)
print(f"Instruments: {INSTRUMENTS}")
print(f"Header events: {header}")

# Forced opening: bar 0 with key / time / tempo committed before free sampling.
forced = [
    bar_event(0),
    key_event(0, KEY_SHARPS),
    time_event(0, TIME_SIG_IDX),
    tempo_event(0, TEMPO_IDX),
]
print(f"Forced opening events: {forced}")

print(f"\nGenerating {N_MEASURES} measures from scratch...")
fill_events = model.generate_standard(
    header_events=header,
    forced_events=forced,
    target_bars=N_MEASURES,
    max_new=MAX_NEW,
    temperature=TEMPERATURE,
    top_k=TOP_K,
    device=DEVICE,
    max_parts=len(instr_indices),
)

print(f"Generated {len(fill_events)} events ({count_bars(fill_events)} bars)")

### Export

In [ ]:
# Reconstruct score from generated events. The forced opening is part of the
# committed prompt in generate_standard(), so it must be prepended here.
full_events = [sos_event()] + header + forced + fill_events + [eos_event()]
score = events_to_score(full_events)

# Color generated notes red for easy identification
for el in score.flatten().notesAndRests:
    if el.isNote or el.isChord:
        el.style.color = 'red'

instr_tag = "_".join(INSTRUMENTS)
name = f"scratch_{instr_tag}_{N_MEASURES}bars"
midi_path = str(OUTPUT_DIR / f"{name}.mid")
xml_path  = str(OUTPUT_DIR / f"{name}.xml")

score.write("midi", fp=midi_path)
print(f"Saved MIDI -> {midi_path}")

try:
    score.write("musicxml", fp=xml_path)
    print(f"Saved MusicXML -> {xml_path}")
except Exception as e:
    print(f"MusicXML export skipped: {e}")
    xml_path = None

note_count   = sum(1 for el in score.flatten().notesAndRests if el.isNote or el.isChord)
measure_count = len(score.parts[0].getElementsByClass("Measure")) if score.parts else 0
print(f"\nScore summary: {measure_count} measures, {note_count} notes/chords")